# Canvas Submission Metadata Triage

Upload the **ZIP downloaded from Canvas** and run the cells in order. The notebook accepts `.xlsx`, `.xlsm`, `.docx`, `.pptx`, and `.pdf`. It creates an evidence-first Excel workbook; unusual metadata is a prompt for manual review, **not proof of AI use or misconduct**.

**What it does:** preserves every submission, deduplicates byte-identical team copies, extracts common Office/PDF metadata, records software fingerprints, calculates a structural signature for OOXML files, and exports a readable report.

In [20]:
# Install the one dependency Colab may not already have.
!pip -q install pypdf

from google.colab import files
uploaded = files.upload()

# Group uploaded files by type
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]

# Supported formats to allow standalone uploads
SUPPORTED_EXTS = {'.xlsx', '.xlsm', '.docx', '.pptx', '.pdf'}
standalone_files = [name for name in uploaded if any(name.lower().endswith(ext) for ext in SUPPORTED_EXTS)]

if not zip_names and not standalone_files:
    raise ValueError('Please upload at least one Canvas ZIP file or supported Office/PDF document.')

print(f'Using {len(zip_names)} ZIP files: {zip_names}')
print(f'Using {len(standalone_files)} direct file uploads: {standalone_files}')

Saving submissions.zip to submissions.zip
Using 1 ZIP files: ['submissions.zip']
Using 0 direct file uploads: []


In [21]:
# Extract all uploaded Canvas archives safely or organize direct file uploads.
from pathlib import Path
import shutil, zipfile

WORK = Path('/content/submission_metadata_triage')
EXTRACT = WORK / 'extracted'
shutil.rmtree(WORK, ignore_errors=True)
EXTRACT.mkdir(parents=True)

# 1. Process ZIP files if uploaded
for zip_name in zip_names:
    zip_path = Path(zip_name)
    # Create a unique subfolder per ZIP file to prevent filename collisions
    zip_extract_dir = EXTRACT / zip_path.stem
    zip_extract_dir.mkdir(parents=True, exist_ok=True)

    print(f'Extracting {zip_path.name}...')
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (zip_extract_dir / member.filename).resolve()
            if not str(target).startswith(str(zip_extract_dir.resolve())):
                raise ValueError(f'Unsafe ZIP path detected in {zip_path.name}: {member.filename}')
            zf.extract(member, zip_extract_dir)

# 2. Process direct file uploads if present
if standalone_files:
    print(f'Copying {len(standalone_files)} direct file upload(s) to working directory...')
    for file_name in standalone_files:
        src_path = Path(file_name)
        dest_path = EXTRACT / src_path.name
        shutil.copy2(src_path, dest_path)

SUPPORTED = {'.xlsx', '.xlsm', '.docx', '.pptx', '.pdf'}
submission_files = sorted(p for p in EXTRACT.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED)
print(f'Found a total of {len(submission_files)} supported files across all sections.')
if not submission_files:
    raise ValueError('No supported Office or PDF files were found.')

Extracting submissions.zip...
Found a total of 28 supported files across all sections.


In [22]:
# Analysis helpers. Feel free to edit the REVIEW_TERMS list for your own context.
import hashlib, json, re, zipfile
from collections import Counter
from datetime import datetime, timezone
from xml.etree import ElementTree as ET
import pandas as pd
from pypdf import PdfReader

NS = {'cp':'http://schemas.openxmlformats.org/package/2006/metadata/core-properties',
      'dc':'http://purl.org/dc/elements/1.1/',
      'dcterms':'http://purl.org/dc/terms/',
      'ep':'http://schemas.openxmlformats.org/officeDocument/2006/extended-properties',
      'r':'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
      'main':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

OOXML = {'.xlsx', '.xlsm', '.docx', '.pptx'}
REVIEW_TERMS = ['openpyxl', 'xlsxwriter', 'python-docx', 'python-pptx', 'libreoffice', 'openoffice', 'google', 'gspread', 'pandas', 'reportlab', 'weasyprint', 'wkhtmltopdf', 'gpt', 'openai', 'claude', 'gemini', 'copilot']

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def clean(value):
    if value is None: return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

def read_xml(zf, name):
    try: return ET.fromstring(zf.read(name))
    except (KeyError, ET.ParseError): return None

def text_at(root, query):
    node = root.find(query, NS) if root is not None else None
    return clean(node.text) if node is not None else ''

def ooxml_info(path):
    meta, evidence, details = {}, [], {}
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        core = read_xml(zf, 'docProps/core.xml')
        app = read_xml(zf, 'docProps/app.xml')
        custom = read_xml(zf, 'docProps/custom.xml')

        fields = {'creator':'dc:creator','last_modified_by':'cp:lastModifiedBy','created':'dcterms:created','modified':'dcterms:modified','title':'dc:title','subject':'dc:subject','description':'dc:description','keywords':'cp:keywords','category':'cp:category'}
        meta.update({key:text_at(core, xpath) for key,xpath in fields.items()})

        meta['application'] = text_at(app, 'ep:Application')
        meta['app_version'] = text_at(app, 'ep:AppVersion')
        meta['company'] = text_at(app, 'ep:Company')
        meta['manager'] = text_at(app, 'ep:Manager')
        meta['total_edit_time'] = text_at(app, 'ep:TotalTime')
        meta['shared_doc'] = text_at(app, 'ep:SharedDoc')

        if custom is not None:
            details['custom_properties'] = ' | '.join(clean(x.text) for x in custom.iter() if x.text)[:3000]

        blob = (' '.join([zf.read(n).decode('utf-8', 'ignore') for n in names if n.endswith(('.xml','.rels'))])).lower()
        for term in REVIEW_TERMS:
            if term in blob:
                evidence.append(f'Package contains: {term}')

        # Format-specific structural observations
        if path.suffix.lower() in {'.xlsx','.xlsm'}:
            # Get sheets and visibility metadata
            wb_xml = read_xml(zf, 'xl/workbook.xml')
            sheets_list = []
            hidden_sheets = []
            very_hidden_sheets = []
            if wb_xml is not None:
                sheets_node = wb_xml.find('main:sheets', NS)
                if sheets_node is not None:
                    for s in sheets_node.findall('main:sheet', NS):
                        name = s.get('name', '')
                        state = s.get('state', 'visible')
                        sheets_list.append(name)
                        if state == 'hidden':
                            hidden_sheets.append(name)
                        elif state == 'veryHidden':
                            very_hidden_sheets.append(name)

            details['sheets'] = ', '.join(sheets_list)
            details['hidden_sheets'] = ', '.join(hidden_sheets) if hidden_sheets else 'None'
            details['very_hidden_sheets'] = ', '.join(very_hidden_sheets) if very_hidden_sheets else 'None'
            details['hidden_parts'] = ', '.join(n for n in names if '/hidden' in n.lower())
            details['has_macros'] = 'Yes' if any(n.lower().endswith('vbaproject.bin') for n in names) else 'No'
            details['has_external_links'] = 'Yes' if any('externalLink' in n for n in names) else 'No'
            details['formula_cells'] = sum(zf.read(n).count(b'<f') for n in names if n.startswith('xl/worksheets/') and n.endswith('.xml'))
            details['tables'] = sum(1 for n in names if n.startswith('xl/tables/'))
            details['charts'] = sum(1 for n in names if n.startswith('xl/charts/'))

        elif path.suffix.lower() == '.docx':
            details['comments'] = sum(1 for n in names if 'comments' in n.lower())
            details['tracked_changes'] = 'Yes' if b'<w:ins' in zf.read('word/document.xml') or b'<w:del' in zf.read('word/document.xml') else 'No'

        elif path.suffix.lower() == '.pptx':
            details['slides'] = sum(1 for n in names if re.match(r'ppt/slides/slide\d+\.xml$', n))

        # Structural Signature
        parts = []
        for n in sorted(names):
            if n.startswith('docProps/') or n.endswith('.rels') or n == '[Content_Types].xml': continue
            if n.endswith('.xml'):
                raw = zf.read(n).decode('utf-8','ignore')
                raw = re.sub(r'\br:id="[^"]+"', '', raw)
                raw = re.sub(r'<dcterms:(created|modified)[^>]*>.*?</dcterms:(created|modified)>', '', raw)
                parts.append(n + ':' + hashlib.sha256(raw.encode()).hexdigest())
        details['structural_signature'] = hashlib.sha256('\n'.join(parts).encode()).hexdigest() if parts else ''
    return meta, evidence, details

def pdf_info(path):
    meta, evidence, details = {}, [], {}
    reader = PdfReader(str(path))
    raw = reader.metadata or {}
    keymap = {'/Author':'creator','/Creator':'application','/Producer':'producer','/Title':'title','/Subject':'subject','/Keywords':'keywords','/CreationDate':'created','/ModDate':'modified'}
    meta = {new: clean(raw.get(old, '')) for old,new in keymap.items()}
    details['pages'] = len(reader.pages)
    details['encrypted'] = 'Yes' if reader.is_encrypted else 'No'
    blob = ' '.join(clean(v) for v in raw.values()).lower()
    for term in REVIEW_TERMS:
        if term in blob:
            evidence.append(f'PDF metadata contains: {term}')
    details['structural_signature'] = hashlib.sha256((str(len(reader.pages)) + str(sorted(raw.keys()))).encode()).hexdigest()
    return meta, evidence, details

def analyze(path):
    digest = sha256(path)
    base = {'relative_path':str(path.relative_to(EXTRACT)), 'file_name':path.name, 'extension':path.suffix.lower(), 'bytes':path.stat().st_size, 'sha256':digest}
    try:
        meta, evidence, details = ooxml_info(path) if path.suffix.lower() in OOXML else pdf_info(path)
        flags = []
        if not meta.get('creator'): flags.append('Creator metadata missing')
        if not meta.get('application') and path.suffix.lower() in OOXML: flags.append('Application metadata missing')
        if evidence: flags.extend(evidence)
        base.update(meta); base.update(details); base['evidence'] = ' | '.join(evidence); base['review_flags'] = ' | '.join(flags); base['analysis_error'] = ''
    except Exception as exc:
        base.update({'evidence':'','review_flags':'Could not fully parse','analysis_error':f'{type(exc).__name__}: {exc}'})
    return base

In [23]:
# Run the triage, calculate AI likelihood scores, and produce the consolidated report.
records = [analyze(path) for path in submission_files]
all_submissions = pd.DataFrame(records)

# Exact duplicates: a 4-person team that uploaded the same bytes remains one unique artifact
all_submissions['duplicate_group_size'] = all_submissions.groupby('sha256')['sha256'].transform('size')
all_submissions['exact_duplicate_group'] = all_submissions.groupby('sha256').ngroup() + 1
unique_files = all_submissions.drop_duplicates('sha256', keep='first').copy()

duplicate_groups = (all_submissions.groupby(['exact_duplicate_group','sha256'], dropna=False)
                    .agg(submission_count=('file_name','size'),
                         submitted_files=('relative_path', lambda x: '\n'.join(x)))
                    .reset_index())
duplicate_groups = duplicate_groups[duplicate_groups['submission_count'] > 1]

# Near-duplicate candidates use an OOXML structural signature
signature_counts = unique_files['structural_signature'].replace('', pd.NA).value_counts()
unique_files['similar_structure_group_size'] = unique_files['structural_signature'].map(signature_counts).fillna(1).astype(int)
near_duplicates = unique_files[(unique_files['similar_structure_group_size'] > 1) & unique_files['structural_signature'].notna()].copy()

# Define the AI Likelihood Scoring logic
def calculate_ai_likelihood(row):
    score = 0
    reasons = []

    # 1. Programmatic software signatures
    evidence_str = str(row.get('evidence', '')).lower()
    prog_tools = ['openpyxl', 'xlsxwriter', 'python-docx', 'python-pptx', 'pandas', 'gspread', 'reportlab', 'weasyprint', 'wkhtmltopdf']
    detected_prog = [tool for tool in prog_tools if tool in evidence_str]
    if detected_prog:
        score += 45
        reasons.append(f"Programmatic tool(s) detected: {', '.join(detected_prog)}")

    # 2. Direct AI footprint in metadata
    ai_tools = ['gpt', 'openai', 'claude', 'gemini', 'copilot']
    detected_ai = [ai for ai in ai_tools if ai in evidence_str]
    if detected_ai:
        score += 35
        reasons.append(f"AI keyword(s) detected: {', '.join(detected_ai)}")

    # 3. Missing standard metadata
    app = str(row.get('application', '')).strip()
    creator = str(row.get('creator', '')).strip()
    ext = str(row.get('extension', '')).lower()

    if ext in ['.xlsx', '.xlsm', '.docx', '.pptx']:
        if not app:
            score += 10
            reasons.append("Missing Application tag (common in raw scripts)")
        if not creator:
            score += 5
            reasons.append("Missing Creator name")

    # 4. Instant or non-human editing timestamps
    try:
        created = str(row.get('created', '')).strip()
        modified = str(row.get('modified', '')).strip()
        if created and modified and created == modified:
            score += 10
            reasons.append("Creation & modification times are identical (instant generation)")
    except:
        pass

    score = min(score, 100)
    return score, '; '.join(reasons) if reasons else 'Normal metadata footprints'

# Calculate scores
scores_data = []
reasons_data = []
for idx, row in unique_files.iterrows():
    s, r = calculate_ai_likelihood(row)
    scores_data.append(s)
    reasons_data.append(r)

unique_files['ai_likelihood_score'] = scores_data
unique_files['ai_likelihood_reasons'] = reasons_data

# Map scores back to all submissions dataset
score_map = unique_files.set_index('sha256')['ai_likelihood_score'].to_dict()
reason_map = unique_files.set_index('sha256')['ai_likelihood_reasons'].to_dict()
all_submissions['ai_likelihood_score'] = all_submissions['sha256'].map(score_map)
all_submissions['ai_likelihood_reasons'] = all_submissions['sha256'].map(reason_map)

# Filter sub-tabs with score details included
metadata_columns = ['file_name','relative_path','sha256','creator','last_modified_by','created','modified','title','subject','description','keywords','category','application','app_version','company','manager','total_edit_time','producer','custom_properties']
metadata_details = unique_files.reindex(columns=[c for c in metadata_columns if c in unique_files.columns])
evidence_columns = ['file_name','relative_path','extension','application','producer','evidence','review_flags','analysis_error','duplicate_group_size','similar_structure_group_size']
flags = unique_files.reindex(columns=[c for c in evidence_columns if c in unique_files.columns])
flags = flags[(flags['evidence'].fillna('') != '') | (flags['review_flags'].fillna('') != '') | (flags['analysis_error'].fillna('') != '')]

# Write consolidated spreadsheet
OUT_SCORED = '/content/Canvas_Submission_Metadata_Triage_Scored_Report.xlsx'
with pd.ExcelWriter(OUT_SCORED, engine='openpyxl') as writer:
    all_submissions.to_excel(writer, sheet_name='All Submissions', index=False)
    unique_files.to_excel(writer, sheet_name='Unique Files', index=False)
    duplicate_groups.to_excel(writer, sheet_name='Exact Duplicate Groups', index=False)
    near_duplicates.to_excel(writer, sheet_name='Similar Structure Leads', index=False)

    # Add metrics metadata detail tab
    meta_cols = list(metadata_details.columns) + ['ai_likelihood_score', 'ai_likelihood_reasons']
    unique_files.reindex(columns=[c for c in meta_cols if c in unique_files.columns]).to_excel(writer, sheet_name='Complete Metadata', index=False)

    # Add flags tab
    flag_cols = list(flags.columns) + ['ai_likelihood_score', 'ai_likelihood_reasons']
    unique_files.reindex(columns=[c for c in flag_cols if c in unique_files.columns]).to_excel(writer, sheet_name='Review Flags', index=False)

    for ws in writer.book.worksheets:
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        for col in ws.columns:
            letter = col[0].column_letter
            ws.column_dimensions[letter].width = min(max(12, max(len(str(c.value or '')) for c in col) + 2), 42)

print(f'Analyzed {len(all_submissions)} submissions; {len(unique_files)} were byte-unique.')
print(f'Scored report generated and written to: {OUT_SCORED}')
files.download(OUT_SCORED)

Analyzed 28 submissions; 28 were byte-unique.
Scored report generated and written to: /content/Canvas_Submission_Metadata_Triage_Scored_Report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

These are additional checkers. The first block checks instances of claude.

In [24]:
# Inspect the extracted XML packages to pinpoint exactly where 'claude' or other review terms are stored.
import zipfile
from pathlib import Path

target_term = 'claude'
found_any = False

for file_path in submission_files:
    if file_path.suffix.lower() in {'.xlsx', '.xlsm', '.docx', '.pptx'}:
        try:
            with zipfile.ZipFile(file_path) as zf:
                for name in zf.namelist():
                    if name.endswith(('.xml', '.rels')):
                        content = zf.read(name).decode('utf-8', 'ignore')
                        if target_term in content.lower():
                            found_any = True
                            print(f"[Match Found]")
                            print(f"File: {file_path.name}")
                            print(f"Internal Zip Archive Path: {name}")

                            # Extract and display the match with surrounding context
                            idx = content.lower().find(target_term)
                            start = max(0, idx - 150)
                            end = min(len(content), idx + 150)
                            context = content[start:end]
                            print(f"Context:\n... {context} ...\n")
        except Exception as e:
            pass

if not found_any:
    print(f"No files containing the literal term '{target_term}' were found in the current run.")

[Match Found]
File: jimenezayden_LATE_146201_23126032_BUS3391_Grade_Calculator_[Jimenez].xlsx.xlsx
Internal Zip Archive Path: xl/webextensions/webextension1.xml
Context:
... eferences><we:reference id="WA200009404" version="1.0.0.8" store="en-US" storeType="omex"/></we:alternateReferences><we:properties><we:property name="claude.fileId" value="&quot;1479093f-283b-462d-91f6-e28afa15a347&quot;"/></we:properties><we:bindings/><we:snapshot xmlns:r="http://schemas.openxmlfor ...



In [25]:
# Inspect and list ALL web extensions (add-ins) and taskpane integrations across all submissions
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

web_extension_results = []

for file_path in submission_files:
    if file_path.suffix.lower() in {'.xlsx', '.xlsm', '.docx', '.pptx'}:
        try:
            with zipfile.ZipFile(file_path) as zf:
                # Web extensions are stored in xl/webextensions/ or word/webextensions/
                ext_files = [n for n in zf.namelist() if 'webextension' in n.lower() and n.endswith('.xml')]
                for ext_file in ext_files:
                    raw_xml = zf.read(ext_file)
                    root = ET.fromstring(raw_xml)

                    # Parse namespaces safely
                    ns = {'we': 'http://schemas.microsoft.com/office/webextensions/webextension/2010/11'}

                    # Extract store reference and properties
                    reference = root.find('.//we:reference', ns)
                    store_id = reference.get('id', 'Unknown ID') if reference is not None else 'Unknown'
                    store_type = reference.get('storeType', 'Unknown Store') if reference is not None else 'Unknown'

                    # Gather properties
                    props = []
                    properties_node = root.find('.//we:properties', ns)
                    if properties_node is not None:
                        for prop in properties_node.findall('.//we:property', ns):
                            name = prop.get('name', '')
                            val = prop.get('value', '')
                            props.append(f"{name}={val}")

                    web_extension_results.append({
                        'file': file_path.name,
                        'archive_path': ext_file,
                        'add_in_id': store_id,
                        'store_type': store_type,
                        'properties': ' | '.join(props) if props else 'None'
                    })
        except Exception as e:
            pass

# Display findings
if web_extension_results:
    print(f"Found {len(web_extension_results)} active Web Add-in(s) in your student submissions:\n")
    ext_df = pd.DataFrame(web_extension_results)
    display(ext_df)
else:
    print("No Web Add-ins (external office extensions) were discovered in any of the current submissions.")

Found 2 active Web Add-in(s) in your student submissions:



,file,archive_path,add_in_id,store_type,properties
0,jimenezayden_LATE_146201_23126032_BUS3391_Grad...,xl/webextensions/taskpanes.xml,Unknown,Unknown,None
1,jimenezayden_LATE_146201_23126032_BUS3391_Grad...,xl/webextensions/webextension1.xml,WA200009404,omex,"claude.fileId=""1479093f-283b-462d-91f6-e28afa1..."


In [26]:
# Advanced analyzer resolving Store IDs to friendly names (e.g. Claude, ChatGPT, Copilot)
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd

# Mapping known Microsoft Office Store IDs (WA-IDs) to human-readable names
KNOWN_ADD_INS = {
    'WA200009404': 'Claude (Anthropic)',
    'WA200008084': 'ChatGPT for Excel / OpenAI',
    'WA104380862': 'Microsoft Copilot / Bing AI',
    'WA104379724': 'Solver Add-in',
    'WA104380121': 'Data Analysis ToolPak (Web)',
    'WA200002422': 'Grammarly for Word'
}

resolved_add_ins = []

for file_path in submission_files:
    if file_path.suffix.lower() in {'.xlsx', '.xlsm', '.docx', '.pptx'}:
        try:
            with zipfile.ZipFile(file_path) as zf:
                ext_files = [n for n in zf.namelist() if 'webextension' in n.lower() and n.endswith('.xml')]
                for ext_file in ext_files:
                    raw_xml = zf.read(ext_file)
                    root = ET.fromstring(raw_xml)

                    ns = {'we': 'http://schemas.microsoft.com/office/webextensions/webextension/2010/11'}
                    reference = root.find('.//we:reference', ns)

                    if reference is not None:
                        store_id = reference.get('id', 'Unknown ID')
                        store_type = reference.get('storeType', 'Unknown Store')

                        # Look up friendly name
                        friendly_name = KNOWN_ADD_INS.get(store_id, f'Other Add-In ({store_id})')

                        # Gather custom configuration keys
                        props = []
                        properties_node = root.find('.//we:properties', ns)
                        if properties_node is not None:
                            for prop in properties_node.findall('.//we:property', ns):
                                props.append(f"{prop.get('name', '')}={prop.get('value', '')}")

                        resolved_add_ins.append({
                            'File Name': file_path.name,
                            'Detected Tool': friendly_name,
                            'Store ID': store_id,
                            'Store Type': store_type,
                            'Linked Session Keys': ' | '.join(props) if props else 'None'
                        })
        except Exception as e:
            pass

if resolved_add_ins:
    print(f"Resolved {len(resolved_add_ins)} add-in instances:")
    report_df = pd.DataFrame(resolved_add_ins)
    display(report_df)
else:
    print("No Office Add-ins found in the uploaded files.")

Resolved 1 add-in instances:


,File Name,Detected Tool,Store ID,Store Type,Linked Session Keys
0,jimenezayden_LATE_146201_23126032_BUS3391_Grad...,Claude (Anthropic),WA200009404,omex,"claude.fileId=""1479093f-283b-462d-91f6-e28afa1..."


In [27]:
# Show every raw, unique Add-In ID discovered in the submissions, regardless of whether it is on our list
KNOWN_ADD_INS = {
    'WA200009404': 'Claude (Anthropic)',
    'WA200008084': 'ChatGPT for Excel / OpenAI',
    'WA104380862': 'Microsoft Copilot / Bing AI',
    'WA104379724': 'Solver Add-in',
    'WA104380121': 'Data Analysis ToolPak (Web)',
    'WA200002422': 'Grammarly for Word'
}

all_discovered_ids = set()
for item in web_extension_results:
    all_discovered_ids.add(item['add_in_id'])

print("Every unique Add-In Store ID detected in these submissions:")
for store_id in sorted(all_discovered_ids):
    friendly = KNOWN_ADD_INS.get(store_id, "Unrecognized/Custom Add-In")
    print(f"- {store_id} ({friendly})")

Every unique Add-In Store ID detected in these submissions:
- Unknown (Unrecognized/Custom Add-In)
- WA200009404 (Claude (Anthropic))


In [28]:
import pandas as pd

# Create a clear mapping of files to their detected extensions and IDs
print("=== WEB ADD-IN AUDIT REPORT BY STUDENT FILE ===\n")

for file_path in submission_files:
    file_name = file_path.name
    # Filter resolved add-ins for this specific file
    matches = [item for item in resolved_add_ins if item['File Name'] == file_name]

    if matches:
        print(f"File: {file_name}")
        for m in matches:
            print(f"  -> Detected Tool: {m['Detected Tool']}")
            print(f"     Store ID:      {m['Store ID']}")
            print(f"     Session Info:  {m['Linked Session Keys']}")
        print("-" * 60)
    else:
        print(f"File: {file_name}")
        print("  -> No Web Add-ins detected.")
        print("-" * 60)

=== WEB ADD-IN AUDIT REPORT BY STUDENT FILE ===

File: aragonhusielhugo_145051_23118031_BUS3391_Grade_Calculator_(Konstantinidou, Fielder, Aragon, Zeising).xlsx
  -> No Web Add-ins detected.
------------------------------------------------------------
File: arianmanesharteen_LATE_145898_23130838_~$BUS3391_Arteen Arian, Angelo, Armaan.xlsx
  -> No Web Add-ins detected.
------------------------------------------------------------
File: axeltonrachelelizabeth_148677_23119874_BUS3391_Grade_Calculator_Axelton_Garcia_Hernandez_Rivera.xlsx
  -> No Web Add-ins detected.
------------------------------------------------------------
File: azimovaskhab_148002_23108113_BUS3391_Grade_Calculator_(Azimov, Lehman, Salama, Corben)-2.pdf
  -> No Web Add-ins detected.
------------------------------------------------------------
File: barneylucyannette_116046_23108173_BUS3391_Grade_Calculator_(Barney).xlsx
  -> No Web Add-ins detected.
------------------------------------------------------------
File: baud

## How to read the report

- **All Submissions** retains every Canvas upload, including team duplicates.
- **Unique Files** is the main review sheet: each SHA-256-exact workbook/document appears once.
- **Exact Duplicate Groups** lists the Canvas paths that contained identical file bytes. That is expected for a team submission.
- **Similar Structure Leads** identifies OOXML files that share a structural signature after volatile metadata is excluded. Treat this only as a lead: common templates and instructor-provided starter files can naturally cluster.
- **Review Flags** preserves the observed basis for any flag. Software names or incomplete metadata can arise from legitimate tools, conversions, accessibility workflows, or resaving.

For best interpretation, compare a flagged submission with the assignment instructions, the expected template, the student/team context, and the work itself. Metadata is useful triage evidence; it cannot establish how a student completed the assignment.